# 02 · Tratamento Silver (Bônus 1)
**Parte 1 — Analisar:** investigar a tabela bronze em busca de problemas e inconsistências.

**Parte 2 — Tratar:** corrigir os problemas encontrados e gravar `workspace.fauna.silver_registros`.

In [0]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
print("Fuso da sessão:", spark.conf.get("spark.sql.session.timeZone"))

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.fauna.bronze_registros")
colunas = [c for c in bronze.columns if not c.startswith("_")]

bronze.select(colunas).printSchema()

## Parte 1 — Analisar

In [0]:
display(bronze.limit(10))

In [0]:
nulos = bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas
])
display(nulos)

In [0]:
total = bronze.count()
ids_unicos = bronze.select("id_registro").distinct().count()

sem_id = [c for c in colunas if c != "id_registro"]
linhas_unicas = bronze.dropDuplicates(sem_id).count()

print(f"Total de registros           : {total}")
print(f"IDs repetidos                : {total - ids_unicos}")

In [0]:
display(
    bronze.groupBy("grupo", "especie")
    .count()
    .orderBy("grupo", F.desc("count"))
)

In [0]:
display(bronze.groupBy("id_camera").count().orderBy("id_camera"))

In [0]:
display(bronze.select(
    F.min("data_hora_inicio").alias("primeiro_registro"),
    F.max("data_hora_inicio").alias("ultimo_registro"),
))

In [0]:
display(bronze.select("duracao_segundos", "individuos").summary())

In [0]:
display(
    bronze.select("especie", "id_camera", "duracao_segundos")
    .orderBy(F.desc("duracao_segundos"))
    .limit(10)
)

### Pontos de Atenção
| # | Problema | Coluna(s) | Impacto nas análises |
|---|---|---|---|
| 1 | Tipo de Dados | data_hora_inicio | Verificar a possível necessidade no futuro de separação da data e hora para analises de horários comuns |
| 2 | Registros com "°C" | temperadura_c | Transforma campo em decimal para calculos matemáticos |
| 3 | Registros com "%" | umidade_pct | Transforma campo em decimal para calculos matemáticos |